# Ranking clásico con TF-IDF

Comparación entre el cálculo manual y `TfidfVectorizer`.

## Parte A: cálculo manual

In [3]:
from pathlib import Path
import re
import math
import numpy as np

carpeta_documentos = Path(
    r"C:\Users\harry\Desktop\Universidad\Semestre 2026-B\Recuperaicon de la Informacion\SRCD2\documentos_100"
)

rutas_documentos = sorted(carpeta_documentos.glob("*/*.txt"))

if not rutas_documentos:
    raise FileNotFoundError(
        f"No se encontraron archivos .txt dentro de las carpetas de: {carpeta_documentos}"
    )

documentos = [
    ruta.read_text(encoding="utf-8", errors="ignore")
    for ruta in rutas_documentos
]
nombres_documentos = [ruta.name for ruta in rutas_documentos]

def tokenizar(texto):
    return re.findall(r"(?u)\b\w\w+\b", texto.lower())

documentos_tokenizados = [tokenizar(documento) for documento in documentos]
vocabulario = sorted({termino for tokens in documentos_tokenizados for termino in tokens})
indice_terminos = {termino: indice for indice, termino in enumerate(vocabulario)}

print(f"Documentos cargados: {len(documentos)}")
print(f"Términos del vocabulario: {len(vocabulario)}")

Documentos cargados: 79
Términos del vocabulario: 171296


In [15]:
# Matriz de frecuencias de términos (TF bruto)
matriz_frecuencias = np.zeros(
    (len(documentos_tokenizados), len(vocabulario)),
    dtype=int
)

for fila, tokens in enumerate(documentos_tokenizados):
    for termino in tokens:
        matriz_frecuencias[fila, indice_terminos[termino]] += 1

print(f"Dimensiones de la matriz: {matriz_frecuencias.shape}")
matriz_frecuencias[:5, :10]

Dimensiones de la matriz: (79, 171296)


array([[ 0,  1,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0, 21,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  1,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  3,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  1,  0,  0,  0,  0,  0,  0,  0,  0]])

In [7]:
# Frecuencia documental: cantidad de documentos que contienen cada término
frecuencia_documental = np.count_nonzero(matriz_frecuencias > 0, axis=0)

frecuencia_documental[:10]

array([ 5, 79,  2,  1,  1,  1,  1,  1,  1,  1])

In [8]:
# Inverse Document Frequency (IDF)
# Se usa la misma fórmula suavizada que TfidfVectorizer:
# idf(t) = log((1 + n_documentos) / (1 + df(t))) + 1

numero_documentos = len(documentos)
idf_manual = np.log(
    (1 + numero_documentos) / (1 + frecuencia_documental)
) + 1

idf_manual[:10]

array([3.59026717, 1.        , 4.28341435, 4.68887945, 4.68887945,
       4.68887945, 4.68887945, 4.68887945, 4.68887945, 4.68887945])

In [9]:
# TF-IDF manual: producto de TF bruto por IDF
tfidf_manual_sin_normalizar = matriz_frecuencias.astype(float) * idf_manual

# Normalización L2 por documento, igual que TfidfVectorizer por defecto
normas = np.linalg.norm(tfidf_manual_sin_normalizar, axis=1, keepdims=True)
normas[normas == 0] = 1
tfidf_manual = tfidf_manual_sin_normalizar / normas

tfidf_manual[:5, :10]

array([[0.00000000e+00, 1.36742186e-05, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 2.19997788e-03, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 1.88659872e-04, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 3.83949167e-04, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 1.86862624e-03, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00]])

## Parte B: cálculo con `TfidfVectorizer`

In [10]:
# Parte B: cálculo con TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w\w+\b",
    smooth_idf=True,
    norm="l2"
)

tfidf_vectorizer = vectorizer.fit_transform(documentos)
vocabulario_vectorizer = vectorizer.get_feature_names_out()
matriz_tfidf_vectorizer = tfidf_vectorizer.toarray()

print(f"Dimensiones de la matriz: {matriz_tfidf_vectorizer.shape}")
matriz_tfidf_vectorizer[:5, :10]

Dimensiones de la matriz: (79, 171296)


array([[0.00000000e+00, 1.36742186e-05, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 2.19997788e-03, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 1.88659872e-04, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 3.83949167e-04, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 1.86862624e-03, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00]])

In [12]:
# Comparación de vocabulario, IDF y matriz TF-IDF

mismo_vocabulario = np.array_equal(
    np.array(vocabulario),
    vocabulario_vectorizer
)

idf_vectorizer = vectorizer.idf_
mismo_idf = np.allclose(
    idf_manual,
    idf_vectorizer,
    rtol=1e-10,
    atol=1e-12
)

misma_matriz_tfidf = np.allclose(
    tfidf_manual,
    matriz_tfidf_vectorizer,
    rtol=1e-10,
    atol=1e-12
)

print(f"¿Mismo vocabulario?: {mismo_vocabulario}")
print(f"¿Mismo IDF?: {mismo_idf}")
print(f"¿Misma matriz TF-IDF?: {misma_matriz_tfidf}")
print(
    f"Diferencia máxima: "
    f"{np.max(np.abs(tfidf_manual - matriz_tfidf_vectorizer)):.12f}"
)

¿Mismo vocabulario?: True
¿Mismo IDF?: True
¿Misma matriz TF-IDF?: True
Diferencia máxima: 0.000000000000
